# ResNet50 (Fixed) - AdamW & Custom DataLoader

In [ ]:
import os
import time
import copy
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torchvision import models

# Check device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"🔹 Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from datasets.Caltech256 import build_dataloader

train_csv_path = '/content/archive/train.csv'
val_csv_path = '/content/archive/val.csv'
test_csv_path = '/content/archive/test.csv'
data_root = '/content/archive/256_ObjectCategories'
batch_size = 32
img_size = 224

print("Initializing DataLoaders...")
train_loader = build_dataloader(
    csv_path=train_csv_path,
    data_root=data_root,
    batch_size=batch_size,
    img_size=img_size,
    is_train=True,
    num_workers=4
)

val_loader = build_dataloader(
    csv_path=val_csv_path,
    data_root=data_root,
    batch_size=batch_size,
    img_size=img_size,
    is_train=False,
    num_workers=4
)

test_loader = build_dataloader(
    csv_path=test_csv_path,
    data_root=data_root,
    batch_size=batch_size,
    img_size=img_size,
    is_train=False,
    num_workers=4
)

dataloaders = {'train': train_loader, 'val': val_loader, 'test': test_loader}
dataset_sizes = {'train': len(train_loader.dataset), 'val': len(val_loader.dataset), 'test': len(test_loader.dataset)}

print(f"Dataset Sizes: {dataset_sizes}")

# Setup CONFIG and Class Names
try:
    # Attempt to scan data_root for class names if it exists
    class_names = sorted([d for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, d))])
except OSError:
    print(f"⚠️ Warning: Could not access {data_root} to list classes. Using placeholder.")
    # Fallback or error depending on requirement. 
    class_names = [str(i) for i in range(257)]

CONFIG = {
    'model_dir': 'models',
    'learning_rate': 0.001,
    'weight_decay': 1e-4,
    'num_epochs': 25,
    'patience': 5,
    'batch_size': batch_size
}
os.makedirs(CONFIG['model_dir'], exist_ok=True)

## Model Setup

In [ ]:
# Load Model from Scratch (No Pretrained)
model = models.resnet50(weights=None)

# Change Final Layer
num_classes = len(class_names)
print(f"Number of classes: {num_classes}")

in_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(in_features, num_classes)
)

model = model.to(device)

# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
# Optimizer: AdamW
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], 
                      weight_decay=CONFIG['weight_decay'])

# Scheduler
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3)

## Training Loop

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, verbose=False, delta=0):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), os.path.join(CONFIG['model_dir'], 'checkpoint.pth'))
        self.val_loss_min = val_loss

def train_model(model, criterion, optimizer, scheduler, num_epochs=25, patience=5):
    since = time.time()
    early_stopping = EarlyStopping(patience=patience, verbose=True)
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            pbar = tqdm(dataloaders[phase], desc=f"{phase.upper()}", leave=False)
            
            for inputs, labels in pbar:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
                pbar.set_postfix({'loss': f"{loss.item():.4f}"})

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())
                scheduler.step(epoch_acc)
                early_stopping(epoch_loss, model)
                
                if early_stopping.early_stop:
                    print("Early stopping")
                    model.load_state_dict(torch.load(os.path.join(CONFIG['model_dir'], 'checkpoint.pth')))
                    return model, history

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        
        if early_stopping.early_stop:
             break

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model, history

In [ ]:
model, history = train_model(model, criterion, optimizer, scheduler, num_epochs=CONFIG['num_epochs'], patience=CONFIG['patience'])

## Evaluation

In [ ]:
print(f"\n💥 FINAL EVALUATION ON TEST SET ({len(test_dataset)} images)")
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in tqdm(dataloaders['test'], desc="TESTING"):
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\nClassification Report (Test Set):")
# Ensure class_names is available
if 'class_names' in globals() and len(class_names) == num_classes:
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))
else:
    print(classification_report(all_labels, all_preds, zero_division=0))

In [ ]:
torch.save(model.state_dict(), os.path.join(CONFIG['model_dir'], 'resnet50_256_object_categories_fixed.pth'))
print("Model saved to ./models/resnet50_256_object_categories_fixed.pth")